# 02 — WOE scorecard

Bins, WOE and coefficients are fitted on train only. The bin **edges** are stored on the fitted object — re-deriving them on validation would give each split its own cut-points, so the fitted WOE values would attach to a different population than they were measured on. That is a quiet form of the same leakage the exclusion list exists to prevent.

In [ ]:
import sys; sys.path.insert(0, '..')
import warnings; warnings.filterwarnings('ignore')
import pandas as pd, numpy as np
from src.config import load_config

cfg = load_config()
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 60)

from src.splits import assign_split
from src.data_load import months_of_credit_history

df = pd.read_parquet('../data/processed/modelling_sample.parquet')
# Elapsed duration, never the raw date: the absolute date tracks vintage
df['credit_history_months'] = months_of_credit_history(df['earliest_cr_line'], df['issue_dt'])
df['split'] = assign_split(df, cfg)
tr, va = df[df['split']=='train'], df[df['split']=='valid']
print(f'train {len(tr):,} | valid {len(va):,}')

## 1. WOE and IV on one feature

Convention (it flips between textbooks, so state it):

$$\\text{WOE}_i = \\ln\\frac{\\text{bad}_i / \\text{bad}_{total}}{\\text{good}_i / \\text{good}_{total}}$$

Positive WOE = this bin is riskier than the population average.

In [ ]:
from src.binning import bin_numeric, woe_table, information_value

bins = bin_numeric(tr['fico_range_low'], max_bins=6)
tab = woe_table(bins, tr['target'])
print(f'IV = {information_value(tab):.4f}')
tab[['bin','count','bad_rate','woe','iv_contrib']]

## 2. IV screening

**Verified finding: no single application-time feature is strong.** The best legitimate one is FICO at IV 0.125; everything else is weak. Meanwhile `sub_grade` scores 0.305 because it is already a composite of these same inputs.

Two consequences: signal must come from *combination*, and the platform benchmark is a real opponent rather than a straw man.

In [ ]:
NUM = ['fico_range_low','annual_inc','dti','bc_open_to_buy','avg_cur_bal','tot_cur_bal',
       'acc_open_past_24mths','total_rev_hi_lim','mort_acc','revol_util','inq_last_6mths',
       'delinq_2yrs','open_acc','total_acc','loan_amnt','mths_since_recent_inq',
       'num_tl_90g_dpd_24m','percent_bc_gt_75','revol_bal','credit_history_months']
CAT = ['home_ownership','purpose','verification_status','emp_length','initial_list_status']

from src.binning import profile_features
profile_features(tr, NUM+CAT, 'target', cfg)[
    ['feature','iv','verdict','monotonic','fill_rate']].head(15)

## 3. Fit, with an IV floor

Full sign-constrained pruning (dropping every non-positive coefficient) was tested and **cost about 34 bp** of policy performance — it removed 14 of 27 features. The shipped model uses the IV floor only and keeps the sign check as a reported diagnostic.

That trade-off is usually asserted. Here it is measured.

In [ ]:
from src.scorecard import Scorecard

probe = Scorecard(NUM+CAT, max_bins=cfg['binning']['max_bins']).fit(tr)
keep = probe.coefficient_table()
keep = keep.loc[keep['iv'] >= cfg['binning']['iv_floor'], 'feature'].tolist()
NUM = [f for f in NUM if f in keep]; CAT = [f for f in CAT if f in keep]

sc = Scorecard(NUM+CAT, max_bins=cfg['binning']['max_bins']).fit(tr)
sc.coefficient_table()

## 4. The points table

This — not the AUC — is what an underwriter executes, a compliance reviewer audits, and an analyst traces when a declined applicant asks why.

In [ ]:
pts = sc.points_table()
pts[pts['feature']=='fico_range_low']